# DataLoader 与短程训练对比

本节只确认 non-greedy → greedy packing 的数据利用率和端到端影响。两条当前可运行路线分别是 causal SDPA + non-greedy，以及 block-causal SDPA + greedy；Attention 算子增量留到 06.06。


In [ ]:
import os
from pathlib import Path

original_dir = Path.cwd()
configured_root = os.environ.get('TORCHTITAN_ROOT')
candidates = [Path(configured_root)] if configured_root else []
for parent in (original_dir, *original_dir.parents):
    candidates.extend((parent / 'torchtitan-npu', parent.parent / 'torchtitan-npu'))
torchtitan_root = next((path.resolve() for path in candidates if (path / 'scripts/run_train.sh').is_file()), None)
if torchtitan_root is None:
    raise RuntimeError('未找到 torchtitan-npu；请设置 TORCHTITAN_ROOT。')
os.chdir(torchtitan_root)
if cann_env := os.environ.get('CANN_ENV_SCRIPT'):
    os.environ['BASH_ENV'] = cann_env
os.environ['PYTHONPATH'] = str(original_dir) + os.pathsep + os.environ.get('PYTHONPATH', '')
print('torchtitan root:', torchtitan_root)


## 固定条件

| 项目 | 配置 |
|---|---|
| 初始权重 | `assets/hf/Qwen3-1.7B`；`checkpoint.enable=True`、`load_only=True` |
| seed / 数据顺序 | `42` / `train[:900]` |
| 固定验证集 | `train[900:]`，两路都强制 non-greedy，完整评估 100 条 |
| 训练 | 2 NPU，seq_len=4096，GBS=64，2 optimizer steps |

第 1 步含首次编译；性能取第 2 步，loss 同时记录 step 1/2。两步实验不启用 Torch Profiler，因为 profiler 需要额外 warmup step；算子 trace 使用 06.06 的 10-step 实验。


In [ ]:
%%bash
set -euo pipefail
HF_DATASETS_OFFLINE=1 NGPU=2 \
MODULE=chapter6_recipes CONFIG=wordle_non_greedy_gbs64 \
bash scripts/run_train.sh \
  --checkpoint.enable \
  --checkpoint.load-only \
  --debug.no-print-config \
  2>&1 | tee /tmp/ch6_non_greedy_gbs64.log


In [ ]:
%%bash
set -euo pipefail
HF_DATASETS_OFFLINE=1 NGPU=2 \
MODULE=chapter6_recipes CONFIG=wordle_greedy_block_gbs64 \
bash scripts/run_train.sh \
  --checkpoint.enable \
  --checkpoint.load-only \
  --debug.no-print-config \
  2>&1 | tee /tmp/ch6_greedy_block_gbs64.log


## 结果口径

TorchTitan 的 GBS/TPS 统计定长 container slots：两路每步都是 $64\times4096$ slots，因此原生 TPS 只表示 slot throughput。Packing 是否有效，还要同时报告每步装入的原始样本数、非 padding token 和 supervised token。

本 notebook 以前嵌入的数值来自旧的 EOS 分段实现。当前实现已经改为：每条样本的位置编号重新从 0 开始，trainer 根据这些起点构造 block-causal mask。旧数值不能继续代表当前路线，因此不在正文保留。

2026-07-29 本轮实测取第 2 步。`non-padding tokens/s`、`supervised tokens/s` 和 `raw samples/step` 均为两卡全局值；raw sample 与 padding 前 token 数按固定 split、数据顺序和 tokenizer 直接计数，不从 EOS 推断。

| 路线 | step time | slot TPS/device | raw samples/step | non-padding tokens/s | supervised tokens/s | validation loss |
|---|---:|---:|---:|---:|---:|---:|
| non-greedy + causal SDPA | 22.174 s | 5,911 | 64 | 3,667 | 2,785 | 0.7985 |
| greedy + block-causal SDPA | 24.363 s | 5,380 | 158 | 8,717 | 6,615 | 1.6020 |

Dense block mask 让单步时间增加 9.9%、slot TPS/device 降低 9.0%，但同一 wall time 内的 non-padding 与 supervised token 吞吐都提高约 2.38 倍，raw sample/s 提高约 2.25 倍。

两步训练只用于 smoke test：第 1 步包含首次编译，第 2 步只能检查数量级。两路每个 optimizer step 消费的数据量不同（第 2 步为 61,746 vs 161,170 supervised tokens），所以不能用上表的 validation loss 按 step 直接判断收敛快慢；稳定收敛实验应固定累计 supervised tokens、raw samples 或 wall time。


In [ ]:
import re
from pathlib import Path

def key_lines(path: str):
    text = Path(path).read_text(errors='replace')
    return re.findall(r'(?:validate )?step:\s*[12].*', text)

for route in ('non_greedy', 'greedy_block'):
    print(route)
    print(*key_lines(f'/tmp/ch6_{route}_gbs64.log'), sep='\n')


## 判断

DataLoader correctness gate 已通过：同一 Wordle sample 内部的多个 EOS 没有产生新边界，下一条 packed sample 开始时 position 重新变为 0。当前结论只支持“greedy packing 提高有效数据吞吐”；1–2 步 validation loss 不支持收敛排名。


In [ ]:
%cd $original_dir


## 练习

1. （判断题）两步短程训练中，第 1 步包含首次编译，因此更适合用第 2 步检查性能数量级。

2. （判断题）TorchTitan 原生 TPS 统计定长 container slots，不能单独说明 packing 后的有效数据吞吐。

3. （单选题）比较 non-greedy 与 greedy packing 时，哪组指标最能补足 slot TPS？
    A. raw samples/s、non-padding tokens/s、supervised tokens/s
    B. 只看模型参数量
    C. 只看第 1 步 loss
    D. 只看 checkpoint 文件大小

4. （判断题）两条路线每个 optimizer step 消费的 supervised token 数不同，因此 1–2 步 validation loss 不能直接用于收敛排名。

In [ ]:
!cat ./answer/06.04_answer.txt
